In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_dir=r'C://Users/kerrie/Documents/02_LocalData/nclimgrid_monthly/'

In [3]:
# info for grid cell selection
city_name = ['San Diego', 'Tucson', 'El Paso', 'Austin', 'Houston', 'New Orleans', 'Mobile', 'Augusta', 'Jacksonville', 'Orlando',]
city_lat = 	[32.810, 32.200, 31.77, 30.27, 29.76, 29.95, 30.68, 33.47, 30.33, 28.500 ]
city_lon = 	[-117.140, -110.890, -106.48, -97.74, -95.36, -90.08, -88.04, -81.97, -81.65, -81.370]
lat_min, lat_max = 28,34
lon_min, lon_max = -117.5,-81

# time vars
year1_start, year2_start, year_end = '1950','1949','2024'
base_start, base_end = '1991','2020'

# Prepare pr file 1 (8 cities)

In [ ]:
# get nclimgrid monthly pr
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year1_start,year_end),lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
ds

In [ ]:
# select single grid for each city, save grid lat/lon, calc anomalies

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat[:-2],city_lon[:-2])):
    print('processing',city_name[i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

print(lat)
print(lon)
print(pr_anom[0])

In [ ]:
# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name[:-2]):
    if i==0:
        df = pd.DataFrame(pr_anom[i].data,index=pr_anom[i].time,columns=[city_name[i]])
    else:
        df[city]= pr_anom[i].data
df

In [ ]:
# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

In [ ]:
# add grid lat/lon info
df_T.insert(loc=0, column='LATITUDE', value=lat[:-2])
df_T.insert(loc=1, column='LONGITUDE', value=lon[:-2])
# add other metadata columns
df_T.insert(loc=2, column='UNITS', value='mm')
df_T.insert(loc=3, column='BASE', value='1991-2020')
df_T.insert(loc=4, column='SOURCE', value='https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332')
df_T.insert(loc=5, column='ACCESSED', value='MARCH 2025')
df_T.head()

In [ ]:
# write csv file
df_T.to_csv('pr_anomaly_nclimgrid_monthly_8cities.csv', index_label='CITY')

# Prepare file 2 (2 cities with different start date)

In [ ]:
# get nclimgrid monthly pr
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year2_start,year_end),lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
ds

In [ ]:
# select single grid for each city, save grid lat/lon, calc anomalies

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat[-2:],city_lon[-2:])):
    print('processing',city_name[-2:][i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

print(lat)
print(lon)

In [ ]:
# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name[-2:]):
    if i==0:
        df = pd.DataFrame(pr_anom[-2:][i].data,index=pr_anom[-2:][i].time,columns=[city_name[-2:][i]])
    else:
        df[city]= pr_anom[-2:][i].data
df

In [ ]:
# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

In [ ]:
# add grid lat/lon info
df_T.insert(loc=0, column='LATITUDE', value=lat[-2:])
df_T.insert(loc=1, column='LONGITUDE', value=lon[-2:])
# add other metadata columns
df_T.insert(loc=2, column='UNITS', value='mm')
df_T.insert(loc=3, column='BASE', value='1991-2020')
df_T.insert(loc=4, column='SOURCE', value='https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332')
df_T.insert(loc=5, column='ACCESSED', value='MARCH 2025')
df_T.head()

In [ ]:
# write csv file
df_T.to_csv('pr_anomaly_nclimgrid_monthly_2cities.csv', index_label='CITY')

# Test on additional cities



In [ ]:
city_name = ['Seattle',	'Portland',	'Boise']
city_lat = [47.608013, 45.5370, 43.618881]
city_lon = [-122.31, -122.6500, -116.215019]

In [ ]:
# ds.prcp.isel(time=0).sel(lat=slice(47.7,47.5),lon=slice(-122.4,-122.3)).plot()


In [ ]:
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year1_start,year_end),lat=slice(48,43.5),lon=slice(-122.5,-116))

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat,city_lon)):
    print('processing',city_name[i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # drop unnecessary coordinates
    city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

# create time-indexed pandas dataframe (columns are pr anomalies by city)
for i,city in enumerate(city_name):
    if i==0:
        df = pd.DataFrame(pr_anom[i].data,index=pr_anom[i].time,columns=[city_name[i]])
    else:
        df[city]= pr_anom[i].data

# transpose so city names are the indexes
df_T = df.T
print(df_T.shape)
df_T.head()

In [ ]:
df


# Create netcdf file of 10 stations

In [7]:
# get nclimgrid monthly pr
ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice('1948','2025'),lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
# ds = xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(lat=slice(lat_max,lat_min),lon=slice(lon_min,lon_max))
ds

<xarray.Dataset> Size: 467MB
Dimensions:  (time: 925, lat: 144, lon: 876)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * lat      (lat) float32 576B 33.98 33.94 33.9 33.85 ... 28.1 28.06 28.02
  * lon      (lon) float32 4kB -117.5 -117.4 -117.4 ... -81.1 -81.06 -81.02
Data variables:
    prcp     (time, lat, lon) float32 467MB ...
Attributes: (12/14)
    date_created:              2025-01-07 12:54:24.863985
    date_modified:             2025-01-07 12:54:24.864077
    Conventions:               CF-1.6, ACDD-1.3
    ncei_template_version:     NCEI_NetCDF_Grid_Template_v2.0
    title:                     nClimGrid
    naming_authority:          gov.noaa.ncei
    ...                        ...
    geospatial_lat_min:        24.562532
    geospatial_lat_max:        49.3542
    geospatial_lon_min:        -124.6875
    geospatial_lon_max:        -67.020836
    geospatial_lat_units:      degrees_north
    geospatial_lon_units:      degrees_east

In [8]:
# select single grid for each city, save grid lat/lon, calc anomalies

lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat,city_lon)):
    print('processing',city_name[i])
    
    # select pr at nearest lat/lon
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    
    # save values of nearest lat/lon
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)

    # # drop unnecessary coordinates
    # city_pr = city_pr.drop_vars(['lat','lon'])

    # calc anomalies
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice(base_start,base_end)).groupby('time.month').mean('time'))

print(lat)
print(lon)
print(pr_anom[0])

processing San Diego
processing Tucson
processing El Paso
processing Austin
processing Houston
processing New Orleans
processing Mobile
processing Augusta
processing Jacksonville
processing Orlando
[array(32.812534, dtype=float32), array(32.187534, dtype=float32), array(31.770866, dtype=float32), array(30.270866, dtype=float32), array(29.770866, dtype=float32), array(29.937532, dtype=float32), array(30.687532, dtype=float32), array(33.4792, dtype=float32), array(30.312532, dtype=float32), array(28.4792, dtype=float32)]
[array(-117.145836, dtype=float32), array(-110.895836, dtype=float32), array(-106.479164, dtype=float32), array(-97.729164, dtype=float32), array(-95.354164, dtype=float32), array(-90.0625, dtype=float32), array(-88.020836, dtype=float32), array(-81.979164, dtype=float32), array(-81.645836, dtype=float32), array(-81.354164, dtype=float32)]
<xarray.DataArray 'prcp' (time: 925)> Size: 4kB
array([-5.45390625e+01, -3.09976273e+01, -4.56695938e+00, -1.31106777e+01,
       -6.

In [9]:
pr_anom[0]

<xarray.DataArray 'prcp' (time: 925)> Size: 4kB
array([-5.45390625e+01, -3.09976273e+01, -4.56695938e+00, -1.31106777e+01,
       -6.23496103e+00,  4.31119800e-01, -8.30371082e-01, -6.22330725e-01,
       -3.44091797e+00,  1.52063475e+01, -2.09447594e+01,  2.11878586e+01,
        4.30703125e+01, -2.06577835e+01, -2.38569984e+01, -1.68304043e+01,
        6.09511709e+00, -1.57864583e+00, -8.30371082e-01, -6.22330725e-01,
       -2.96044922e+00, -7.15400410e+00,  6.00543594e+00, -1.88424149e+01,
        2.42607422e+01, -2.98775101e+01, -1.54370766e+01, -1.08899746e+01,
        9.44726467e-01, -1.53860676e+00,  7.39941418e-01, -6.22330725e-01,
       -3.06103516e+00, -1.29635744e+01,  1.14253578e+01, -4.11422195e+01,
       -5.51953125e+00, -4.47583694e+01, -2.95962563e+01,  2.82799473e+01,
       -6.69492197e+00, -1.57864583e+00,  2.59472668e-01,  4.36790371e+00,
       -8.20800781e-01,  1.50061522e+01,  9.70465469e+00,  7.61380539e+01,
        7.19003906e+01, -5.43980179e+01,  8.97728882e+01,  2.06197910e+01,
       -6.81503916e+00,  5.12369871e-02, -7.70800769e-01, -6.22330725e-01,
       -2.60107422e+00, -1.30836916e+01,  3.37749672e+01,  1.85277023e+01,
       -3.57597656e+01, -5.13384476e+01, -1.96265297e+01, -4.62044334e+00,
       -1.66464853e+00,  8.21744800e-01, -6.39941394e-01, -6.22330725e-01,
       -3.44091797e+00, -1.20436525e+01, -2.14495468e+00, -4.27818680e+01,
        3.70205078e+01, -3.73482132e+01,  8.70336304e+01, -1.67405605e+01,
       -5.86484385e+00, -9.88802075e-01,  7.89746106e-01, -5.92057288e-01,
...
       -3.44091797e+00, -1.38349628e+00,  5.65485001e+00,  1.91478195e+01,
        5.95019531e+00,  6.15717087e+01, -9.56695938e+00, -1.23899746e+01,
        1.52650394e+01,  3.81315112e-01, -2.50292957e-01, -6.22330725e-01,
       -5.30761719e-01, -1.30836916e+01,  5.38853188e+01,  4.88177414e+01,
       -4.56689453e+01, -5.27779007e+01,  3.68431969e+01,  7.23600235e+01,
       -5.98496103e+00,  3.32174492e+00, -8.30371082e-01, -6.22330725e-01,
       -3.42138672e+00, -1.17536135e+01, -5.35491562e+00, -2.67515945e+01,
       -5.26953125e+00, -6.29282913e+01,  4.79339218e+00, -1.57805996e+01,
       -5.15488291e+00, -9.58528638e-01,  7.69238293e-01,  1.80735683e+00,
        2.34912109e+00,  1.60862312e+01, -2.24349937e+01,  3.66077805e+01,
       -5.10292969e+01, -4.67779007e+01, -1.16656876e+00, -1.47698574e+01,
       -6.48496103e+00, -1.49856770e+00, -8.10839832e-01, -2.32682288e-01,
        7.79931641e+00, -7.31416035e+00,  8.99469376e+00, -2.48206329e+00,
        9.13505859e+01, -1.07075882e+01,  8.74437866e+01, -1.66302090e+01,
       -1.48496103e+00,  1.41080737e-01, -7.70800769e-01,  3.45774746e+00,
       -1.79052734e+00, -1.12633791e+01, -4.70452499e+00, -2.59420242e+01,
        2.35703125e+01,  4.84623337e+01,  3.33334312e+01, -1.12005215e+01,
       -3.81503916e+00, -1.19876301e+00, -8.30371082e-01,  5.68098962e-01,
       -3.07080078e+00, -1.30036135e+01, -1.84047203e+01, -4.45719070e+01,
       -4.71191406e+01], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
    lat      (time) float32 4kB 32.81 32.81 32.81 32.81 ... 32.81 32.81 32.81
    lon      (time) float32 4kB -117.1 -117.1 -117.1 ... -117.1 -117.1 -117.1
    month    (time) int64 7kB 1 2 3 4 5 6 7 8 9 10 11 ... 4 5 6 7 8 9 10 11 12 1

In [10]:
for i,arr in enumerate(pr_anom):
    arr = arr.drop_vars(['lat','lon','month'])
    pr_anom[i]=arr
    

In [11]:
subset = xr.concat(pr_anom,dim='location')
subset

<xarray.DataArray 'prcp' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
Dimensions without coordinates: location

In [12]:
location = xr.DataArray(city_name, coords={'location':('location',city_name)})
subset.coords['location']=location

subset

<xarray.DataArray 'prcp' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'

In [13]:
subset = xr.DataArray(subset, coords={'time':('time',subset.time.data),'location':('location',city_name),'lat':('location',lat),'lon':('location',lon)})
subset

<xarray.DataArray 'prcp' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'
    lat       (location) float32 40B 32.81 32.19 31.77 ... 33.48 30.31 28.48
    lon       (location) float32 40B -117.1 -110.9 -106.5 ... -81.65 -81.35

In [14]:
subset.name = 'pr_anom'

subset

<xarray.DataArray 'pr_anom' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'
    lat       (location) float32 40B 32.81 32.19 31.77 ... 33.48 30.31 28.48
    lon       (location) float32 40B -117.1 -110.9 -106.5 ... -81.65 -81.35

In [20]:
attrs_to_keep = ['standard_name','long_name','units']

subset.location.attrs = {'standard_name':'location','long_name':'Location Name', 'description':'string place name associated with a single lat/lon grid cell'}
subset.time.attrs = {k:ds.time.attrs[k] for k in attrs_to_keep if k in ds.time.attrs}
subset.lat.attrs = {k:ds.lat.attrs[k] for k in attrs_to_keep if k in ds.lat.attrs}
subset.lon.attrs ={k:ds.lon.attrs[k] for k in attrs_to_keep if k in ds.lon.attrs}
subset.attrs = {k:ds.prcp.attrs[k] for k in attrs_to_keep if k in ds.prcp.attrs}

subset

<xarray.DataArray 'pr_anom' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'
    lat       (location) float32 40B 32.81 32.19 31.77 ... 33.48 30.31 28.48
    lon       (location) float32 40B -117.1 -110.9 -106.5 ... -81.65 -81.35
Attributes:
    standard_name:  precipitation_amount
    long_name:      Precipitation, monthly total
    units:          millimeter

In [21]:
subset.attrs['standard_name'] = 'pr_anom'
subset.attrs['long_name'] = 'Precipitation Anomaly'
subset.attrs['description'] = 'monthly total precipitation departure from monthly long term mean over the 30-year base period 1991-2020'
subset

<xarray.DataArray 'pr_anom' (location: 10, time: 925)> Size: 37kB
array([[-54.539062 , -30.997627 ,  -4.5669594, ..., -18.40472  ,
        -44.571907 , -47.11914  ],
       [-26.734375 ,  15.8083   ,  -1.3423824, ...,  -2.8490238,
        -29.77435  , -24.094727 ],
       [ -5.231348 ,   8.133073 ,  -4.761165 , ...,  10.365202 ,
        -14.309668 , -10.251856 ],
       ...,
       [ 11.524673 ,  68.11673  , 117.41016  , ...,  47.461098 ,
        -39.898537 , -35.885483 ],
       [ 63.682777 , -35.19352  , 182.43863  , ..., -30.60521  ,
        -32.650032 ,  72.74235  ],
       [ 98.9946   , -22.266472 ,  17.004555 , ..., -21.253971 ,
        -24.752995 , -22.575714 ]], shape=(10, 925), dtype=float32)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'
    lat       (location) float32 40B 32.81 32.19 31.77 ... 33.48 30.31 28.48
    lon       (location) float32 40B -117.1 -110.9 -106.5 ... -81.65 -81.35
Attributes:
    standard_name:  pr_anom
    long_name:      Precipitation Anomaly
    units:          millimeter
    description:    monthly total precipitation departure from monthly long t...

In [22]:
ds_out = subset.to_dataset()
ds_out.attrs = {'source':'NOAA NClimGrid Monthly', 
                'data_access':'https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00332',
                'date_accessed':'Feb 2025',
               'reference':'GHCN-Monthly Version 3 (Vose et al. 2011)'}
ds_out

<xarray.Dataset> Size: 45kB
Dimensions:   (time: 925, location: 10)
Coordinates:
  * time      (time) datetime64[ns] 7kB 1948-01-01 1948-02-01 ... 2025-01-01
  * location  (location) <U12 480B 'San Diego' 'Tucson' ... 'Orlando'
    lat       (location) float32 40B 32.81 32.19 31.77 ... 33.48 30.31 28.48
    lon       (location) float32 40B -117.1 -110.9 -106.5 ... -81.65 -81.35
Data variables:
    pr_anom   (location, time) float32 37kB -54.54 -31.0 ... -24.75 -22.58
Attributes:
    source:         NOAA NClimGrid Monthly
    data_access:    https://www.ncei.noaa.gov/access/metadata/landing-page/bi...
    date_accessed:  Feb 2025
    reference:      GHCN-Monthly Version 3 (Vose et al. 2011)

In [24]:
subset.to_netcdf('../nclimgrid/pr_anom_nclimgrid_monthly_10cities.nc')

In [ ]:
# base period for these anomalies is 1991-2020
enso = pd.read_csv(r'../NOAA_ClimateIndex/Nino34ClimateIndex.txt',
                   header=None,
                   skiprows=1,
                   skipfooter=3,
                   na_values=-99.99,
                   delimiter=r'\s+',
                   names=np.arange(1,13),
                   engine='python')
enso

In [ ]:
enso = enso.dropna(how='any')
enso.shape

In [ ]:
enso_arr = enso.to_numpy().flatten()
type(enso_arr), enso_arr.shape

In [ ]:
# enso.describe()

In [ ]:
year_start = str(enso.index[0])
year_end = str(enso.index[-1])
year_start,year_end

In [ ]:
# cities to pull out grid cells for
# city_name = ['San Diego','Tucson','San Antonio','Orlando','Nashville','Louisville','Columbus','St. Louis']
# city_lat = 	[32.810, 32.200, 29.460, 28.500, 36.170, 38.220, 39.990, 38.627]
# city_lon = 	[-117.140, -110.890, -98.510, -81.370, -86.780, -85.740, -82.990, -90.199]

city_name = ['San Diego', 'Tucson', 'El Paso', 'Austin', 'Houston', 'New Orleans', 'Mobile', 'Augusta', 'Jacksonville', 'Orlando',]
city_lat = 	[32.810, 32.200, 31.77, 30.27, 29.76, 29.95, 30.68, 33.47, 30.33, 28.500 ]
city_lon = 	[-117.140, -110.890, -106.48, -97.74, -95.36, -90.08, -88.04, -81.97, -81.65, -81.370]

In [ ]:
# ds.prcp.isel(time=0).sel(lat=slice(31.8,31.7),lon=slice(-106.55,-106.4)).plot()

In [ ]:
# get precip
ds=xr.open_dataset(data_dir+'nclimgrid_prcp.nc').sel(time=slice(year_start,year_end),lat=slice(34,28),lon=slice(-117.5,-81))
ds

# Subset to only 8 cities and calc anomalies

In [ ]:
# city_pr = ds.prcp.sel(lat=city_lat[0],lon=city_lon[0],method='nearest')
# lat = city_pr.lat.data
# lon = city_pr.lon.data
# city_pr = city_pr.drop_vars(['lat','lon'])
# pr_anom = city_pr.groupby('time.month') - city_pr.sel(time=slice('1991','2020')).groupby('time.month').mean('time')
# pr_anom
# # city_pr

In [ ]:
# lat

In [ ]:
lat=[] # list of nearest lat point
lon=[] # list of nearest lon point
pr_anom =[] # list of xr data array timeseries objects for each city

for i,(y,x) in enumerate(zip(city_lat,city_lon)):
    print('processing',city_name[i])
    city_pr = ds.prcp.sel(lat=y,lon=x,method='nearest')
    lat.append(city_pr.lat.data) 
    lon.append(city_pr.lon.data)
    city_pr = city_pr.drop_vars(['lat','lon'])
    # pr_anom.append(city_pr)
    pr_anom.append(city_pr.groupby('time.month') - city_pr.sel(time=slice('1991','2020')).groupby('time.month').mean('time'))

print(lat)
print(lon)
print(pr_anom[0])

In [ ]:
df = pd.DataFrame(pr_anom[0].data,index=pr_anom[0].time,columns=[city_name[0]])
df

In [ ]:
for i,city in enumerate(city_name[1:]):
    df[city]= pr_anom[i+1].data

df

In [ ]:
data = {'LATITUDE':city_lat, 'LONGITUDE':city_lon}
city_info = pd.DataFrame(data,index=city_name)
city_info

In [ ]:
df_T = df.T
print(df_T.shape)
df_T.head()

In [ ]:
df_T.insert(loc=0, column='LATITUDE', value=city_info.LATITUDE)
df_T.head()

In [ ]:
df_T.insert(loc=1, column='LONGITUDE', value=city_info.LONGITUDE)
df_T.head()

In [ ]:
df_T['LATITUDE']=lat
df_T['LONGITUDE']=lon

df_T.head()

In [ ]:
df1 = df_T.loc['San Diego':'Augusta']
df2 = df_T.loc['Jacksonville':'Orlando']

df1.shape,df2.shape

In [ ]:
enso_arr = enso.values.flatten()
enso_arr.shape

In [ ]:
df['enso']=enso_arr
df

In [ ]:
# df['San Diego norm']=(df['San Diego']-df['San Diego'].min())/(df['San Diego'].max()-df['San Diego'].min())
# df['San Diego SD']=(df['San Diego']-df['San Diego'].mean())/df['San Diego'].std()

# df[['San Diego SD','enso']].plot()

In [ ]:
# df[['San Diego','enso']].corr()

In [ ]:
# pr_arr = df['San Diego'].values
# type(pr_arr), pr_arr.shape


In [ ]:
nino_inds = np.where(enso_arr>=1)[0]
nina_inds = np.where(enso_arr<=-1)[0]
neut_inds = np.where( (enso_arr>=-.25)&(enso_arr<=.25) )[0]
nino_inds.shape, nina_inds.shape,neut_inds.shape

In [ ]:
# pr_arr[nino_inds].mean(),pr_arr[nina_inds].mean(),pr_arr[neut_inds].mean()

In [ ]:
for city in city_name:
    pr_arr=df[city].values
    print(city,pr_arr[nino_inds].mean(),pr_arr[nina_inds].mean(),pr_arr[neut_inds].mean(),', nino-nina=',round(pr_arr[nino_inds].mean()-pr_arr[nina_inds].mean(),2))    

Calculate the difference between mean nino and mean nina precip anomalies for each station
Plot the differences with the xaxis labeled with the city names

Sort stations by nino-nina pr difference value highest to lowest

Plot the sorted differences on a bar plot with xaxis city names


In [ ]:
pr_arr = df.loc[:,'San Diego':'Orlando'].values
pr_arr.shape

In [ ]:
nino_mean = pr_arr[nino_inds,:].mean(axis=0)
nina_mean = pr_arr[nina_inds,:].mean(axis=0)
neut_mean = pr_arr[neut_inds,:].mean(axis=0)
ninoa_diff = nino_mean-nina_mean
ninoa_diff

In [ ]:
# which city has the largest mean precip anomaly during el nino months?
print(city_name[nino_mean.argmax()])

# which city has the largest mean precip anomaly during la nina months?
print(city_name[nina_mean.argmax()])

# which city has the largest difference between mean precip anomalies in nino vs nina months?
print(city_name[ninoa_diff.argmax()])


In [ ]:
fig = plt.figure(figsize=(8,3))
plt.plot(city_name,ninoa_diff,marker='o',lw=0)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (mm)')
plt.title('nino minus nina mean monthly precip difference')
plt.show()

In [ ]:
sort_order = np.argsort(ninoa_diff)
sorted_diff = ninoa_diff[sort_order]
sorted_labels = np.array(city_name)[sort_order]

print(sorted_diff)
print(sorted_labels)

In [ ]:
fig = plt.figure(figsize=(8,3))
plt.bar(sorted_labels,sorted_diff)
plt.xticks(rotation=300,ha='left')
plt.ylabel('Precipitation (mm)')
plt.title('nino minus nina mean monthly precip difference')
plt.show()

In [ ]:
# ds.prcp.sel(lat=lat[1],lon=lon[1]).drop_vars(['lat','lon']).to_dataframe()


In [ ]:
# step 9 spatial subset
# clip data to a bounding box

# get clip object
clipobj=gpd.read_file(shpfile)
clipobj.crs

In [ ]:
# assign crs to netcdf data
ds.rio.write_crs("epsg:4326",inplace=True)
ds_clip=ds.rio.clip(clipobj.geometry.apply(shapely.geometry.mapping),clipobj.crs,drop=True,invert=False)

In [ ]:
ds_clip['tmax'] = ds_clip.tmax.round(decimals=2)
ds_clip

In [ ]:
del ds_clip['spatial_ref']
ds_clip

In [ ]:
ds_clip = ds_clip.reindex(lat=ds_clip.lat[::-1])
ds_clip

In [ ]:
tmax=ds_clip.tmax.data.astype('float16')
tmax.shape, tmax.nbytes

In [ ]:
lat= list(ds_clip.lat.data)
lon= list(ds_clip.lon.data)
time=ds_clip.time.data
# lat

In [ ]:
time = time.astype('str')
time = [t[0:7] for t in time]
time[0]

In [ ]:
with open('nclimgrid_tmax_196401-202312.npy','wb') as f:
    np.save(f,tmax)

with open('nclimgrid_tmax_meta_lat.txt','w') as f:
    np.savetxt(f,lat,fmt='%.6f')

with open('nclimgrid_tmax_meta_lon.txt','w') as f:
    np.savetxt(f,lon,fmt='%.6f')

with open('nclimgrid_tmax_meta_time.txt','w') as f:
    np.savetxt(f,time,fmt='%s')

In [ ]:
test=np.load('nclimgrid_tmax.npy')
test.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.contourf(test[0,:,:])

In [ ]:
# ds_clip.tmax.to_netcdf(data_dir+'nclimgrid_tmax_196401-202312.nc')
ds_clip.tmax.to_netcdf(data_dir+'nclimgrid_tmax_199401-202312.nc')

In [ ]:
import numpy as np
              # C:\Users\kerrie\Documents\01_LocalCode\repos\DEV_canvas_beginner_py
datafile = r'C://Users/kerrie/Documents/01_LocalCode/repos/DEV_canvas_beginner_py/tmax_data.csv'

In [ ]:
# data = np.loadtxt(datafile, delimiter=',', skiprows=1)
data = np.loadtxt('tmax_data.csv', delimiter=',', skiprows=1, usecols=(3), unpack=True)


data

In [ ]:
# 1) drop data we don't need"
ds = ds.drop_vars('tavg')

steps 2-7 aren't necessary, data already looks good wrt to these items

In [ ]:
# step 8 millimeter --> mm/day
ds.prcp.attrs['units']='mm/day'

In [ ]:
# step 9 spatial subset
# clip data to a bounding box

# get clip object
box=gpd.read_file(shpfile)

# assign crs to netcdf data
ds.rio.write_crs("epsg:4326",inplace=True)
ds_clip=ds.rio.clip(box.geometry.apply(shapely.geometry.mapping),box.crs,drop=True,invert=False)
ds_clip

In [ ]:
# save metadata separately for later
coords = ds_clip.coords
prcp_attrs = ds_clip.prcp.attrs
tmax_attrs = ds_clip.tmax.attrs
tmin_attrs = ds_clip.tmin.attrs
dims = ds_clip.dims
print(dims)
coords

step 10 & 11, round and reduce precision

In [ ]:
# choosing a single time to test data precision, looking for values on order of at least 100
testtime='2000-06-4'
ds_clip.prcp.sel(time=testtime).plot()

In [ ]:
# float16 is probably not enough precision, let's check on a subset of the data

print('loading float16')
prcp_16 = ds_clip.prcp.sel(time=testtime).astype('float16').load()
print('loading float32')
prcp_32 = ds_clip.prcp.sel(time=testtime).astype('float32').load()
print('loading float64')
prcp_64 = ds_clip.prcp.sel(time=testtime).load()

prcp_16.max().item(),prcp_32.max().item(),prcp_64.max().item()

so we can change the data type from float64 to float32 but not go any smaller

In [ ]:
# steps 10 & 11
ds = ds_clip.round(decimals=2)
ds = ds_clip.astype('float32')
ds

# write files

I think what we have to do to write each chunk to a separate file is:
- chunk xr arrays in space instead of time
- convert to numpy so we can use to_delayed and ravel
- every worker needs the xr metadata for variable and coordinates
- write a dask delayed function that takes the numpy array data and the metadata separately
- inside the dask delayed function re-create the xarray object and write to file

In [ ]:
# create an integer index for the dim we will chunk (longitude)
ilon_ind = np.arange(0,len(ds.lon)).astype('int')
ds.coords['ilon_index']=('lon',ilon_ind)
# ds.ilon_index.attrs['standard_name']='integer_index_longitude'

In [ ]:
# chunking along longitude only
nlons=6
ds = ds.chunk({'time':-1,'lat':-1,'lon':nlons})
ds

In [ ]:
ilon_chunks = xr.DataArray(ds.ilon_index.data,coords={'ilon_index':('lon',ds.ilon_index.data)})#.chunk({'lon':nlons}).data.to_delayed().ravel()
ilon_chunks

In [ ]:
# a function to return a list of data chunks 
# and the corresponding longitude coord chunks
def xr_ds_to_delayed(ds,varname):
    # chunk the appropriate variable in ds, delay, ravel to list
    var_chunks = ds[varname].data.to_delayed().ravel()
    # xarray doesn't allow chunking of coordinates, so we have to make a new variable to chunk
    # the convoluted process is xarray-->numpy-->xarray-->chunk-->numpy-->delay-->ravel
    lon_chunks = xr.DataArray(ds.lon.data,coords={'lon':('lon',ds.lon.data)}).chunk({'lon':nlons}).data.to_delayed().ravel()
    ilon_chunks = xr.DataArray(ds.ilon_index.data,coords={'ilon_index':('lon',ds.ilon_index.data)}).chunk({'lon':nlons}).data.to_delayed().ravel()
    return var_chunks,lon_chunks,ilon_chunks,ds.spatial_ref

# numpy back to xarray, reattaching metadata and writing chunks to separate files
def write_chunk_to_netcdf(datapath,chunk_id,varname,
                          np_datachunk,xr_time,xr_lat,
                          spatial_ref,np_ilonind,np_lonchunk,
                          lon_meta,var_meta):
    
    chunk_id = str(chunk_id).zfill(3)
    # numpy-->xarray
    xr_datachunk = xr.Dataset({varname:(['time','lat','lon'],np_datachunk)},
                              coords={'time':('time',xr_time.data),
                                      'lat':('lat',xr_lat.data),
                                      'lon':('lon',np_lonchunk),
                                      'spatial_ref':('spatial_ref',spatial_ref),
                                      'ilon_index':('lon',np_ilonind)})
    # copy over xr metadata
    xr_datachunk.time.attrs=xr_time.attrs
    xr_datachunk.lat.attrs=xr_lat.attrs
    xr_datachunk.lon.attrs=lon_meta
    xr_datachunk.ilon_index.attrs['standard_name']='integer_index_longitude'
    xr_datachunk[varname].attrs=var_meta
    
    # clean up metadata
    attrslist=['time','lat','lon',varname]
    for att in attrslist:
        if 'valid_min' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['valid_min']
        if 'valid_max' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['valid_max']
        if (att!=varname) and ('comment' in xr_datachunk[att].attrs):
            del xr_datachunk[att].attrs['comment']
        if 'id' in xr_datachunk[att].attrs:
            del xr_datachunk[att].attrs['id']            
    
    # write file
    xr_datachunk.to_netcdf(datapath+'chunkedlon/'+varname+'_nClimGridDaily_USsouth_'+chunk_id+'.nc')
    return chunk_id

In [ ]:
%%time 
var='prcp'
var_chunks,lon_chunks,ilon_chunks,spatial_ref = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                spatial_ref,ilonchunk,lonchunk,
                                                ds.lon.attrs,ds[var].attrs) \
            for id,(datachunk,ilonchunk,lonchunk) in enumerate(zip(var_chunks,ilon_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)
len(completed_files)

In [ ]:
%%time 
var='tmax'
var_chunks,lon_chunks = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                lonchunk,ds.lon.attrs,ds[var].attrs) \
            for id,(datachunk,lonchunk) in enumerate(zip(var_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)
len(completed_files)

In [ ]:
%%time 
var='tmin'
var_chunks,lon_chunks = xr_ds_to_delayed(ds,var)
task_list= [dask.delayed(write_chunk_to_netcdf)(data_dir,id,var,
                                                datachunk,ds.time,ds.lat,
                                                lonchunk,ds.lon.attrs,ds.prcp.attrs) \
            for id,(datachunk,lonchunk) in enumerate(zip(var_chunks,lon_chunks))]
completed_files = dask.compute(*task_list)

In [ ]:
var='prcp'
files = glob.glob(data_dir+var+'_nClimGridDaily_USsouth_*.nc')
test=xr.open_mfdataset(files)
test

In [ ]:
test[var].isel(time=15).plot()

# old code below to write 1 single file per variable

In [ ]:
%%time
filename='prcp_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.prcp.to_netcdf(data_dir+filename)

In [ ]:
%%time
filename='tmax_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.tmax.to_netcdf(data_dir+filename)

In [ ]:
%%time
filename='tmin_nClimGridDaily_1951-2024_USsouth.nc'
print('writing',filename)
ds.tmin.to_netcdf(data_dir+filename)

In [ ]:
test=xr.open_mfdataset(data_dir+filename)
test

In [ ]:
client.shutdown()